# مدل IFC با بردار جابجایی واقعی هر اتم سوپرسل (نسخه‌ی نهایی، v3 extraction)

## پیشینه
- **Notebook 16**: خروجی یک‌جای کل ماتریس IFC (8,72,3,3)=15552 — MAE=1.2253
- **Notebook 17**: خروجی factorized با `nn.Embedding(72, dim)` دلخواه — MAE=2.4008 (بدتر)
- **تست‌های تشخیصی 18 (v1 تا v3)**: مسیر زیر طی شد تا تابع استخراج بردار جابجایی درست شود:
  - v1: تطبیق بر اساس norm درون هر گروه — فقط برای `supercell_dim=(1,1,9)` کار می‌کرد (یک‌بعدی)
  - v2: تطبیق بر اساس بردار کامل fractional — ۱۰ از ۱۱ ماده با خطای "عدم تطابق کلید
    (کم=۱، زیاد=۱)" شکست خوردند (خطای مرزی گرد کردن: ۰.۹۹۹۹۹۹۷ گاهی به ۱.۰۰۰ و گاهی به
    ۰.۰۰۰ گرد می‌شد)
  - v3: رفع خطای مرزی با یک `mod` و گرد کردن دوم — **۱۱ از ۱۱ ماده موفق شدند** ✅

## یافته‌ی مهم از تست v3
بردارهای جابجایی استخراج‌شده برای تمام ۱۱ ماده‌ی تستی (که همه `supercell_dim=(3,3,1)`
داشتند) **دقیقاً یکسان** بودند. این یعنی این بردار صرفاً هندسه‌ی شبکه‌ی سوپرسل را توصیف
می‌کند (که بین مواد با ابعاد یکسان، طبیعتاً یکسان است)، نه یک سیگنال اختصاصی هر ماده.

با این حال، نسبت به `nn.Embedding` تصادفی Notebook 17، این بردار از ابتدا رابطه‌ی فیزیکی
صحیح بین تصاویر (کدام تصویر به مرجع نزدیک‌تر است) را در خود دارد، بدون نیاز به یادگیری از
صفر. این نسخه آن فرضیه را روی کل دیتاست (۳۵۸ ماده) آزمایش می‌کند.

In [ ]:
!pip install -q phonopy torch_geometric pymatgen torch-cluster torch-sparse torch-scatter torch-spline-conv -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("Installed")

In [15]:
!pip install -q phonopy torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.7 MB/s eta 0:00:0000:01


In [16]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from sklearn.metrics import mean_absolute_error
from tqdm.notebook import tqdm

from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

device = torch.device('cpu')
print(f"Device: {device}")

Device: cpu


## مسیرهای داده

In [21]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = 'OK' if os.path.exists(path) else 'MISSING'
    print(f"[{exists}]  {name}  ->  {path}")

[OK]  FC_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C
[OK]  POSCAR_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar
[OK]  BANDS_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C
[OK]  ELASTIC_FILE  ->  /kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json
[OK]  FEATURE_CSV  ->  /kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv


## کشف خودکار Supercell صحیح (مستقل برای هر ماده) — بدون تغییر نسبت به نوت‌بوک‌های قبلی

In [22]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    masses = [p['mass'] for p in data['points']]
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc

    return {
        'phonon': phonon,
        'symbols': symbols,
        'frac_coords': frac_coords,
        'lattice': lattice,
        'masses': masses,
        'supercell_dim': dim,
        'dim_match_error': err,
        'force_constants': fc,
        'all_q_points': [p['q-position'] for p in data['phonon']],
        'all_real_freqs': np.array([sorted([b['frequency'] for b in p['band']]) for p in data['phonon']]),
    }

print("find_correct_supercell_dim and load_material_with_phonopy ready")

find_correct_supercell_dim and load_material_with_phonopy ready


## تابع استخراج بردار جابجایی fractional (نسخه‌ی v3، تأیید‌شده روی ۱۱ ماده)

تطبیق بر اساس بردار کامل fractional (با wrap-around دوگانه برای رفع خطای مرزی گرد کردن
۰.۰۰۰ در مقابل ۱.۰۰۰)، نه فقط norm — چون norm به‌تنهایی در شبکه‌های دوبعدی مبهم است.

In [23]:
def extract_image_displacement_vectors(phonon, n_atoms, n_images, round_decimals=3):
    """
    Extract the real fractional displacement vector of each supercell image.
    Matches images across the n_atoms unit-cell-atom groups using the full
    fractional displacement vector (not just its norm, which is ambiguous in
    2D/3D supercells), with double wrap-around to fix boundary rounding
    errors (0.000 vs 1.000).
    Returns: (n_images, 3) array, or (None, reason) on failure.
    """
    supercell = phonon.supercell
    sc_cart = supercell.positions
    sc_lattice = supercell.cell
    s2u_map = supercell.s2u_map

    try:
        inv_lattice = np.linalg.inv(sc_lattice)
    except np.linalg.LinAlgError:
        return None, "singular lattice"

    unique_reps = np.unique(s2u_map)
    if len(unique_reps) != n_atoms:
        return None, "group count mismatch"

    groups = {rep: np.where(s2u_map == rep)[0] for rep in unique_reps}
    if not all(len(v) == n_images for v in groups.values()):
        return None, "group size mismatch"

    per_group_frac = {}
    for rep in unique_reps:
        members = groups[rep]
        disp_cart = sc_cart[members] - sc_cart[rep]
        disp_frac = disp_cart @ inv_lattice
        disp_frac = np.mod(disp_frac, 1.0)
        disp_frac = np.round(disp_frac, decimals=round_decimals)
        # second wrap-around to fold boundary-rounded 1.000 back to 0.000
        disp_frac = np.mod(disp_frac, 1.0)
        disp_frac = np.round(disp_frac, decimals=round_decimals)
        per_group_frac[rep] = disp_frac

    first_rep = unique_reps[0]
    reference_keys = [tuple(v) for v in per_group_frac[first_rep]]
    ref_keys_set = set(reference_keys)

    for rep in unique_reps[1:]:
        this_keys = set(tuple(v) for v in per_group_frac[rep])
        if this_keys != ref_keys_set:
            return None, "key mismatch across groups"

    sorted_keys = sorted(ref_keys_set)
    if len(sorted_keys) != n_images:
        return None, "unique key count mismatch"

    return np.array(sorted_keys, dtype=np.float32), "OK"

print("extract_image_displacement_vectors (v3) ready")

extract_image_displacement_vectors (v3) ready


## ساخت دیتاست کامل — بدون فیلتر اتم (شکل ثابت: 8×72×3×3) + بردار جابجایی هر ماده

In [24]:
POSCAR_DIR_PATH = Path(POSCAR_DIR)
FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}
poscar_files = {f.stem: f for f in POSCAR_DIR_PATH.glob('*.psc')}

common = sorted(set(fc_files) & set(band_yaml_files) & set(poscar_files))
print(f"Number of common materials: {len(common)}")

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

Number of common materials: 358


In [25]:
raw_samples = []
failed_phonopy = []
failed_shape = []
failed_displacement = []
failed_other = []

N_ATOMS_FIXED = 8       # from Notebook 16 part A
N_SUPERCELL_FIXED = 72  # from Notebook 16 part A
N_IMAGES_FIXED = N_SUPERCELL_FIXED // N_ATOMS_FIXED  # 9

displacement_fail_reasons = Counter()

for formula in tqdm(common, desc="Loading materials + extracting displacement vectors"):
    try:
        with open(band_yaml_files[formula]) as f:
            quick_check = yaml.safe_load(f)
        if quick_check['natom'] != N_ATOMS_FIXED:
            failed_shape.append(formula)
            continue

        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            failed_phonopy.append(formula)
            continue

        if result['force_constants'].shape != (N_ATOMS_FIXED, N_SUPERCELL_FIXED, 3, 3):
            failed_shape.append(formula)
            continue

        disp_vectors, status = extract_image_displacement_vectors(
            result['phonon'], N_ATOMS_FIXED, N_IMAGES_FIXED)
        if disp_vectors is None:
            failed_displacement.append(formula)
            displacement_fail_reasons[status] += 1
            continue

        positions_cart = result['frac_coords'] @ result['lattice']

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           N_ATOMS_FIXED,
            'lattice':           result['lattice'].astype(np.float32),
            'positions':         positions_cart.astype(np.float32),
            'atom_elements':     result['symbols'],
            'masses':            np.array(result['masses'], dtype=np.float32),
            'force_constants':   result['force_constants'].astype(np.float32),  # (8, 72, 3, 3)
            'supercell_dim':     result['supercell_dim'],
            'image_displacements': disp_vectors,  # (9, 3) fractional
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          result['all_real_freqs'].astype(np.float32),
            'phonon_obj':        result['phonon'],
        })
    except Exception as e:
        failed_other.append((formula, str(e)))

print(f"\nSucceeded: {len(raw_samples)}")
print(f"Failed (shape mismatch with (8,72,3,3)): {len(failed_shape)}")
print(f"Failed (supercell discovery): {len(failed_phonopy)}")
print(f"Failed (displacement extraction): {len(failed_displacement)}")
print(f"  reasons: {dict(displacement_fail_reasons)}")
print(f"Failed (other errors): {len(failed_other)}")

dims_found = Counter(s['supercell_dim'] for s in raw_samples)
print(f"\nDiscovered supercell_dim distribution:")
for dim, count in dims_found.most_common():
    print(f"   {dim}: {count} materials")

Loading materials + extracting displacement vectors:   0%|          | 0/358 [00:00<?, ?it/s]

/tmp/ipykernel_58/1518532777.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
/usr/local/lib/python3.12/dist-packages/phonopy/api_phonopy.py:278: UserWarning: Warning: Point group symmetries of supercell and primitivecell are different.
  self._search_primitive_symmetry()



Succeeded: 358
Failed (shape mismatch with (8,72,3,3)): 0
Failed (supercell discovery): 0
Failed (displacement extraction): 0
  reasons: {}
Failed (other errors): 0

Discovered supercell_dim distribution:
   (3, 3, 1): 358 materials


## تقسیم Train / Val / Test (seed=42) — مشابه نوت‌بوک‌های قبلی برای مقایسه‌ی منصفانه

In [26]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

Train: 286 | Val: 35 | Test: 37


## ساخت گراف (Bond + Atom)

In [27]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"Atomic features: {N_ATOM_FEATURES} -> {feature_cols}")


def structure_to_bond_graph(positions):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)


def structure_to_atom_graph(atom_elements, positions):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

print("Graph construction functions ready")

Atomic features: 7 -> ['atomic_weight', 'atomic_radius', 'atomic_radius_rahm', 'covalent_radius_cordero', 'vdw_radius', 'atomic_volume', 'lattice_constant']
Graph construction functions ready


In [28]:
bond_graphs, atom_graphs, ifc_targets, image_disp_targets = [], [], [], []

for s in tqdm(raw_samples, desc="Building graphs"):
    bond_graphs.append(structure_to_bond_graph(s['positions']))
    atom_graphs.append(structure_to_atom_graph(s['atom_elements'], s['positions']))
    ifc_targets.append(s['force_constants'])           # (8, 72, 3, 3)
    image_disp_targets.append(s['image_displacements']) # (9, 3) fractional

print(f"Built graphs for {len(bond_graphs)} samples")

Building graphs:   0%|          | 0/358 [00:00<?, ?it/s]

Built graphs for 358 samples


## معماری مدل: دقیقاً مثل Notebook 17، با query از بردار جابجایی واقعی (نه embedding دلخواه)

یک query مجزا برای هر یک از ۷۲ اتم سوپرسل ساخته می‌شود (دقیقاً مثل Notebook 17)، اما هر
query به‌جای یک بردار یادگیری‌شده‌ی دلخواه، از تکرار بردار جابجایی fractional واقعی همان
تصویر می‌آید (با `repeat_interleave` روی ۸ اتم داخل آن تصویر).

In [29]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphRealDisplacementIFCNet(nn.Module):
    """
    Predicts the full IFC tensor using a per-supercell-atom query built from the
    real fractional displacement vector of that atom's image (repeated 8x within
    each image), instead of Notebook 17's free nn.Embedding(72, dim).
    Output shape: (batch, n_atoms_unitcell, n_supercell, 3, 3) = (batch, 8, 72, 3, 3)
    """
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1,
                 n_atoms_unitcell=8, n_images=9, disp_emb_dim=64):
        super().__init__()
        self.n_atoms_unitcell = n_atoms_unitcell
        self.n_images = n_images
        self.n_supercell = n_images * n_atoms_unitcell  # 72
        hidden = 128 if n_atom_features <= 10 else 96

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(5)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(5)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(2)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        # NOTE: Set2Set output is 2*hidden, not hidden -- context_dim accounts for that
        self.context_dim = 2*hidden + hidden + hidden + 2*hidden + hidden + hidden + hidden // 4

        self.disp_emb_dim = disp_emb_dim
        self.displacement_encoder = nn.Sequential(
            nn.Linear(3, 32), nn.SiLU(),
            nn.Linear(32, disp_emb_dim), nn.SiLU()
        )

        per_atom_in_dim = self.context_dim + disp_emb_dim
        per_atom_out_dim = n_atoms_unitcell * 3 * 3  # 8*3*3=72, full IFC column for one supercell atom

        self.ifc_head = nn.Sequential(
            nn.Linear(per_atom_in_dim, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(0.15),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.SiLU(), nn.Dropout(0.15),
            nn.Linear(128, per_atom_out_dim)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data, image_displacements):
        """image_displacements: (batch, n_images, 3) -- real fractional displacement vector per image."""
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)  # 2*hidden
        bond_mean = self.mean_pool(x_bond, bond_data.batch)        # hidden
        bond_max = self.max_pool(x_bond, bond_data.batch)          # hidden

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)  # 2*hidden
        atom_mean = self.mean_pool(x_atom, atom_data.batch)        # hidden
        atom_max = self.max_pool(x_atom, atom_data.batch)          # hidden

        global_features = self.global_mlp(bond_data.u)             # hidden//4

        context = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)  # (batch, context_dim)

        batch_size = context.shape[0]

        # each of the 72 supercell atoms inherits the displacement vector of its image
        # (8 consecutive atoms per image -> repeat_interleave along the image axis)
        disp_per_supercell_atom = image_displacements.repeat_interleave(
            self.n_atoms_unitcell, dim=1)  # (batch, n_images, 3) -> (batch, 72, 3)

        disp_q = self.displacement_encoder(disp_per_supercell_atom)  # (batch, 72, disp_emb_dim)

        context_exp = context.unsqueeze(1).expand(-1, self.n_supercell, -1)  # (batch, 72, context_dim)

        combined_per_atom = torch.cat([context_exp, disp_q], dim=-1)  # (batch, 72, context+disp_emb)
        combined_flat = combined_per_atom.reshape(batch_size * self.n_supercell, -1)

        ifc_flat = self.ifc_head(combined_flat)  # (batch*72, 72)
        ifc_pred = ifc_flat.view(batch_size, self.n_supercell, self.n_atoms_unitcell, 3, 3)
        ifc_pred = ifc_pred.permute(0, 2, 1, 3, 4)  # -> (batch, n_atoms_unitcell, 72, 3, 3)
        return ifc_pred

print("DualGraphRealDisplacementIFCNet defined")

DualGraphRealDisplacementIFCNet defined


## Physics-Informed Loss — ASR (مشابه نوت‌بوک‌های قبلی، برای مقایسه‌ی منصفانه)

In [30]:
def acoustic_sum_rule_loss_full(ifc_pred):
    """ASR over the full matrix (batch, 8, 72, 3, 3): sum over the supercell axis (axis=2) should be zero."""
    asr = torch.sum(ifc_pred, dim=2)
    return torch.mean(asr ** 2)


LAMBDA_ASR = 0.1


def compute_batch_loss_full(model, bond_data, atom_data, ifc_target_batch, disp_batch, device):
    ifc_pred = model(bond_data, atom_data, disp_batch)  # (batch, 8, 72, 3, 3)
    ifc_true = torch.tensor(np.stack(ifc_target_batch), dtype=torch.float32, device=device)

    mse_loss = F.mse_loss(ifc_pred, ifc_true)
    asr_loss = acoustic_sum_rule_loss_full(ifc_pred)

    total_loss = mse_loss + LAMBDA_ASR * asr_loss
    return total_loss, mse_loss.item(), asr_loss.item(), ifc_pred

print(f"Combined loss ready: lambda_asr={LAMBDA_ASR}")

Combined loss ready: lambda_asr=0.1


## حلقه‌ی آموزش

In [31]:
def make_batches(indices, batch_size, shuffle=False):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = np.random.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch_full(model, indices, optimizer=None, batch_size=8, shuffle=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_mse, total_asr = 0., 0., 0.
    n_batches = 0

    for batch_idx in make_batches(indices, batch_size, shuffle):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        ifc_list = [ifc_targets[i] for i in batch_idx]
        disp_list = [image_disp_targets[i] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)
        disp_batch = torch.tensor(np.stack(disp_list), dtype=torch.float32, device=device)  # (batch, 9, 3)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            loss, mse_v, asr_v, _ = compute_batch_loss_full(
                model, bond_batch, atom_batch, ifc_list, disp_batch, device)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        total_mse += mse_v
        total_asr += asr_v
        n_batches += 1

    n_batches = max(n_batches, 1)
    return total_loss/n_batches, total_mse/n_batches, total_asr/n_batches

print("run_epoch_full ready")

run_epoch_full ready


In [32]:
# quick shape sanity check before full training
_test_bond = Batch.from_data_list(bond_graphs[:2]).to(device)
_test_atom = Batch.from_data_list(atom_graphs[:2]).to(device)
_test_disp = torch.tensor(np.stack(image_disp_targets[:2]), dtype=torch.float32, device=device)

_test_model = DualGraphRealDisplacementIFCNet(
    n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1,
    n_atoms_unitcell=8, n_images=9, disp_emb_dim=64).to(device)

with torch.no_grad():
    _test_out = _test_model(_test_bond, _test_atom, _test_disp)

print(f"Model output shape: {_test_out.shape} (expected (2, 8, 72, 3, 3))")
assert _test_out.shape == (2, 8, 72, 3, 3), "Output shape mismatch!"
print("Shape check passed, proceeding to training")
del _test_model, _test_out, _test_bond, _test_atom, _test_disp

Model output shape: torch.Size([2, 8, 72, 3, 3]) (expected (2, 8, 72, 3, 3))
Shape check passed, proceeding to training


In [33]:
model = DualGraphRealDisplacementIFCNet(n_bond_features=6, n_atom_features=N_ATOM_FEATURES,
                                          edge_dim=1, n_atoms_unitcell=8, n_images=9,
                                          disp_emb_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

EPOCHS = 500
PATIENCE = 100
BATCH_SIZE_TRAIN = 8

best_val_loss = float('inf')
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_mse, train_asr = run_epoch_full(
        model, train_idx, optimizer=optimizer, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loss, val_mse, val_asr = run_epoch_full(
        model, val_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, 'best_real_displacement_ifc_model_v3.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
              f"(MSE={val_mse:.4f}, ASR={val_asr:.4f}) Best: {best_val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

Epoch    0 | Train: 5.1314 | Val: 0.8933 (MSE=0.7899, ASR=1.0345) Best: 0.8933
Epoch   10 | Train: 0.8997 | Val: 0.8001 (MSE=0.7897, ASR=0.1042) Best: 0.7985
Epoch   20 | Train: 0.8859 | Val: 0.7939 (MSE=0.7896, ASR=0.0426) Best: 0.7939
Epoch   30 | Train: 0.8805 | Val: 0.7919 (MSE=0.7893, ASR=0.0266) Best: 0.7916
Epoch   40 | Train: 0.8706 | Val: 0.7823 (MSE=0.7783, ASR=0.0407) Best: 0.7817
Epoch   50 | Train: 0.8610 | Val: 0.7744 (MSE=0.7725, ASR=0.0190) Best: 0.7744
Epoch   60 | Train: 0.8585 | Val: 0.7716 (MSE=0.7674, ASR=0.0421) Best: 0.7703
Epoch   70 | Train: 0.8508 | Val: 0.7674 (MSE=0.7626, ASR=0.0479) Best: 0.7656
Epoch   80 | Train: 0.8461 | Val: 0.7625 (MSE=0.7587, ASR=0.0380) Best: 0.7608
Epoch   90 | Train: 0.8412 | Val: 0.7525 (MSE=0.7504, ASR=0.0208) Best: 0.7525
Epoch  100 | Train: 0.8370 | Val: 0.7499 (MSE=0.7466, ASR=0.0332) Best: 0.7497
Epoch  110 | Train: 0.8306 | Val: 0.7457 (MSE=0.7407, ASR=0.0502) Best: 0.7435
Epoch  120 | Train: 0.8279 | Val: 0.7513 (MSE=0.7372

## ارزیابی نهایی — با IFC کاملاً پیش‌بینی‌شده (بدون قرض از Ground Truth)

مقایسه‌ی مستقیم با Notebook 16 و 17.

In [34]:
model.eval()
all_freq_mae = []

with torch.no_grad():
    for i in tqdm(test_idx, desc="Final evaluation (full IFC, no ground-truth borrowing)"):
        s = raw_samples[i]
        bond_batch = Batch.from_data_list([bond_graphs[i]]).to(device)
        atom_batch = Batch.from_data_list([atom_graphs[i]]).to(device)
        disp_batch = torch.tensor(image_disp_targets[i], dtype=torch.float32, device=device).unsqueeze(0)  # (1,9,3)

        ifc_pred_full = model(bond_batch, atom_batch, disp_batch)[0].cpu().numpy()  # (8, 72, 3, 3)

        try:
            phonon = s['phonon_obj']
            phonon.force_constants = ifc_pred_full
            phonon.run_qpoints([[0, 0, 0]])
            pred_freqs = phonon.get_qpoints_dict()['frequencies'][0]

            true_peak = float(np.max(s['y_phonon']))
            pred_peak = float(np.max(np.abs(pred_freqs)))
            all_freq_mae.append(abs(true_peak - pred_peak))
        except Exception as e:
            continue

freq_mae_thz = np.mean(all_freq_mae) if all_freq_mae else float('nan')
freq_mae_cm = freq_mae_thz * 33.35641 if not np.isnan(freq_mae_thz) else float('nan')

print(f"\nFrequency MAE (full IFC, real displacement vectors, no GT borrowing, cm⁻¹): {freq_mae_cm:.4f}")
print(f"   Notebook 11 baseline (direct peak regression):       {0.429 * 33.35641:.4f}")
print(f"   v1-v3 (incorrect manual formula):                    ~{17.2 * 33.35641:.1f}")
print(f"   v4 (real Phonopy, reference block only, upper bound): {0.909 * 33.35641:.4f}")
print(f"   Notebook 16 (full IFC, single-shot, 15552-dim output): {1.2253 * 33.35641:.4f}")
print(f"   Notebook 17 (factorized, free embedding):              {2.4008 * 33.35641:.4f}")
print(f"   This notebook (factorized, real displacement query):  {freq_mae_cm:.4f}")

Final evaluation (full IFC, no ground-truth borrowing):   0%|          | 0/37 [00:00<?, ?it/s]

/tmp/ipykernel_58/1913378463.py:17: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  pred_freqs = phonon.get_qpoints_dict()['frequencies'][0]



Frequency MAE (full IFC, real displacement vectors, no GT borrowing, cm⁻¹): 159.6680
   Notebook 11 baseline (direct peak regression):       14.3099
   v1-v3 (incorrect manual formula):                    ~573.7
   v4 (real Phonopy, reference block only, upper bound): 30.3210
   Notebook 16 (full IFC, single-shot, 15552-dim output): 40.8716
   Notebook 17 (factorized, free embedding):              80.0821
   This notebook (factorized, real displacement query):  159.6680


## جمع‌بندی

این نسخه معماری Notebook 17 را دقیقاً تکرار می‌کند (یک query مجزا برای هر یک از ۷۲ اتم
سوپرسل) — تنها تفاوت این است که هر query از **بردار جابجایی fractional واقعی همان تصویر**
ساخته می‌شود، نه از یک `nn.Embedding(72, dim)` دلخواه.

نکته‌ی مهمی که در مسیر تشخیص باگ کشف شد: بردارهای جابجایی استخراج‌شده برای مواد با
`supercell_dim` یکسان (مثلاً همه‌ی مواد `(3,3,1)`)، دقیقاً یکسان هستند — یعنی این بردار
فقط هندسه‌ی شبکه‌ی سوپرسل را توصیف می‌کند، نه یک سیگنال اختصاصی هر ماده. با این حال، چون
این بردار رابطه‌ی فیزیکی صحیح بین تصاویر (فاصله‌ی واقعی هر تصویر از مرجع) را از ابتدا دارد
(بدون نیاز به یادگیری از صفر مثل embedding دلخواه)، احتمال دارد همچنان به مدل کمک کند.

اگر MAE این نسخه به‌طور قابل‌توجهی بهتر از Notebook 17 (۲.۴۰۰۸) شود، فرضیه‌ی "کیفیت سیگنال
ورودی query" تأیید می‌شود. اگر بهبودی حاصل نشد، باید نتیجه گرفت مشکل عمیق‌تر است (مثلاً
کمبود داده برای یادگیری هر نگاشتی به فضای IFC کامل با ۳۵۸ نمونه) و باید به‌سمت
augmentation فیزیکی یا بازگشت به معماری v4 (با تمرکز روی بهبودهای دیگر) رفت.

### ثبت در Obsidian
نتیجه‌ی این Notebook (MAE نهایی + مقایسه با Notebook 16/17 + درصد مواد رد‌شده در استخراج
بردار جابجایی) باید در `08 - نقشه راه مقاله.md` به‌عنوان آخرین آزمایش بهبود فاز ۳ پروژه
ثبت شود.